# **The Look E-Commerce: End-to-End ETL Pipeline & Star-Schema Data Warehouse**

**Author:** Ryan Farid Pratama <br>
**Tech Stack:** Python, PySpark, Google BigQuery, PostgreSQL, SQL

**Project Overview** <br>
This project builds an ETL pipeline and star-schema data warehouse for **The Look**, a fictional e-commerce clothing retailer. Raw transactional data is extracted from Google BigQuery, cleaned and transformed into dimension and fact tables using PySpark, and loaded into a PostgreSQL data warehouse to support revenue, category, brand, and department analytics.

**Pipeline Architecture** <br>
1. **Extract:** Query raw `order_items`, `products`, and `users` tables from Google BigQuery (`bigquery-public-data.thelook_ecommerce`).
2. **Transform:** Clean missing values, standardize data types, compute profit margins, and model tables using PySpark.
3. **Model:** Design a Star Schema (`fact_sales`, `dim_product`, `dim_users`) optimized for OLAP queries.
4. **Load:** Provision PostgreSQL tables via DDL scripts and load transformed datasets via JDBC.

# A. Business Context & Problem Statement

**Context:**  
As a Data Engineer at an e-commerce company named **The Look**, I need to provide a clear analytical foundation explaining the business need for sales data analysis, as requested by the executive team. To achieve this goal efficiently without slowing down operational databases, building a dedicated Data Warehouse for sales analytics is both necessary and critical.

**Business Problem:**  
Executives plan to create a new targeted/niche marketing strategy by optimizing investment in high-performing products (profitable ones). In order to achieve this, executives require clear insights of high-perform product categories and high-performing brands to optimize marketing campaigns and improve profit margins.

**Objective:**  
Identify which product categories, brands, and departments generate the highest sales volume and revenue.

**Business Process:**  
When a customer purchases an item on the platform, an entry/row is created in "order_items". Each item links to the products list, consist of its name, brand, category, department, cost, and retail price. Aggregating these transactions allows the business to measure total revenue, product costs, and gross profit across different product hierarchies.

# B. Data Modeling & Data Dictionaries
This section documents the data modeling thought process using the Kimball dimensional modeling approach.

### B.1 **Grain**
What does one row in the fact table represent?<br>
1. Grain: `id` alias `order_item_id` (on order_item table) that represents one specific item.
2. Reason: each time an `order_item_id` is created, it captures one individual item purchase along with its associated sales information in the same row.

### B.2 **Fact Table**
What numeric measures are calculated at this grain?
1. Stores numerical sales and financial metrics calculated at the `order_item_id` grain
2. Table Name: `order_item` alias `fact_sale`
3. Primary Key: `id` alias `order_item_id`, 
4. Foreign Key: `product_id`, `user_id`
5. Measure: 
    * `sale_price`: Sales revenue generated for the item
    * `cost`: Unit cost of the product.
    * `profit`: Gross profit calculated as ("sale_price - cost").

#### B.2.1 `fact_sales`
| Column Name | Data Type | Key | Nullable | Description |
| :--- | :--- | :--- | :--- | :--- |
| `order_item_id` | `INTEGER` | PK | No | Unique identifier for each line item in an order. |
| `product_id` | `INTEGER` | FK | No | Foreign key referencing `dim_product (product_id)`. |
| `user_id` | `INTEGER` | FK | No | Foreign key referencing `dim_users (user_id)`. |
| `sale_price` | `NUMERIC(10, 2)` | - | No | Final selling price of the item in USD. |
| `cost` | `NUMERIC(10, 2)` | - | No | Unit manufacturing/acquisition cost in USD. |
| `profit` | `NUMERIC(10, 2)` | - | No | Derived net profit per item (`sale_price - cost`) in USD. |

### B.3 **Dimensions Table**

#### B.3.1 `dim_product`
1. Table Name: `products` alias `dim_products`
2. Primary Key: `product_id`
3. Attribute: `name`, `category`, `brand`, `department`, `retail_price`

| Column Name | Data Type | Key | Nullable | Description |
| :--- | :--- | :--- | :--- | :--- |
| `product_id` | `INTEGER` | PK | No | Unique identifier for each product SKU. |
| `name` | `VARCHAR(255)` | - | No | Full commercial name of the product. |
| `category` | `VARCHAR(100)` | - | No | Product category (e.g., Jeans, Tops & Tees, Outerwear). |
| `brand` | `VARCHAR(100)` | - | No | Brand name of the apparel item. |
| `department` | `VARCHAR(50)` | - | No | Target gender/department ("Men" or "Women"). |

#### B.3.2 `dim_users`
1. Table Name: `users` alias `dim_users`
2. Primary Key: `users_id`
3. Attribute: `gender`, `country`, `traffic_source`

| Column Name | Data Type | Key | Nullable | Description |
| :--- | :--- | :--- | :--- | :--- |
| `user_id` | `INTEGER` | PK | No | Unique identifier for each registered customer. |
| `gender` | `VARCHAR(10)` | - | Yes | Customer gender (`M` or `F`). |
| `country` | `VARCHAR(100)` | - | No | Customer country of residence. |
| `traffic_source` | `VARCHAR(50)` | - | No | Acquisition channel (e.g., Search, Organic, Email, Facebook). |

### B.4 **Schema Diagram Structure**
Schema Diagram Structure The schema used is a Star Schema, with one central fact_sales table referencing two dimension tables (dim_product and dim_users). This design is chosen for the following reasons:
1. Query performance for analytics (OLAP). The business question such as ranking categories, brands, and departments by revenue and profit requires aggregating the fact table and joining to dimensions. A star schema keeps these joins shallow (fact → dimension, one hop), which is faster than the multi-level joins a fully normalized (snowflake) model would require.
2. Simplicity and readability. Analysts can write straightforward JOIN queries between the fact and a single dimension without navigating intermediate bridge tables. This lowers the barrier for the executive/analytics team to self-serve.
3. Separation of measures and context. Numeric measures (sale_price, cost, profit) live in the fact table at a single grain, while descriptive attributes (category, brand, department, country) live in dimensions. This makes the model easy to extend — new attributes can be added to a dimension without touching the fact table.
4. Fit for the grain. Each fact row is one order item, and each dimension describes one product or one user. The star shape maps directly onto this one-fact-many-dimensions relationship.

# C. Environment & Setup
Import the libraries and initialize the Spark session used throughout the pipeline.

In [8]:
# 1. Libraries import & environment setup
import os, sys                                          # Access OS environment variables and Python system paths
import warnings                                         # Control Python warning messages
warnings.filterwarnings("ignore")                       # Suppress non-critical warnings for cleaner notebook output

from dotenv import load_dotenv                          # Load environment variables from local .env file
load_dotenv()                                           # Inject .env variables (DB credentials, GCP keys) into runtime environment

from pyspark.sql import SparkSession                    # Initialize and configure the PySpark session
from pyspark.sql import functions as F                  # Built-in PySpark SQL functions for column transformations
from pyspark.sql.window import Window                   # Window specifications for ranking and analytical partitioning
from pyspark.sql.functions import col, count, when      # Column expressions for null checks and conditional aggregations

In [9]:
# 2. Create SparkSession
spark = SparkSession.builder.appName("Python Spark PostgreSQL").config("spark.jars", "path to jbdc driver .jar file").getOrCreate()

# D. Data Extraction

**Source Dataset:**  
Raw transactional and master data is sourced from the public Google BigQuery dataset **`bigquery-public-data.thelook_ecommerce`** ([BigQuery Console Link](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=thelook_ecommerce)).

**Extraction Strategy:**  
To optimize query performance and minimize scan costs, only the specific columns required by the target Star Schema (`fact_sales`, `dim_product`, and `dim_users`) are extracted via SQL queries in BigQuery and exported as local `.csv` staging files for downstream PySpark processing.

## D.1 Extract Raw Order Items Data

* **Purpose:** Extract line-item order transactions, pricing, and order status to populate the `fact_sales` table.
* **Source Table:** `bigquery-public-data.thelook_ecommerce.order_items`
* **Output Staging File:** `order_items.csv`

```sql
SELECT 
    id AS order_item_id,
    order_id,
    user_id,
    product_id,
    sale_price,
    status
FROM "bigquery-public-data.thelook_ecommerce.order_items";
```

## D.2 Extract Raw Products Data
* **Purpose:** Extract product identity and pricing attributes to populate the `dim_products` table.
* **Source Table:** `bigquery-public-data.thelook_ecommerce.products`
* **Output Staging File:** `products.csv`

```sql
SELECT
  id AS product_id,
  name,
  category,
  brand,
  department,
  cost,
  retail_price
FROM "bigquery-public-data.thelook_ecommerce.products";
```

## D.3 Extract Raw Users Data
* **Purpose:** Extract customer identity and acquisition attributes to populate the `dim_users` table.
* **Source Table:** `bigquery-public-data.thelook_ecommerce.users`
* **Output Staging File:** `users.csv`

```sql
SELECT
  id AS users_id,
  gender,
  country,
  traffic_source
FROM "bigquery-public-data.thelook_ecommerce.users";
```

## D.4 Raw Data Loading
In this section, raw data extracted from BigQuery is loaded into PySpark DataFrame for transformation

In [10]:
# 1. Data Loading Raw Order Items into PySpark DataFrame
df_order_items = spark.read.csv("raw_fact_order_items.csv", header=True, inferSchema=True)

In [11]:
# 2. Data Loading Raw Products into PySpark DataFrame
df_products = spark.read.csv("raw_dim_products.csv", header=True, inferSchema=True)

In [12]:
# 3. Data Loading Raw Users into PySpark DataFrame
df_users = spark.read.csv("raw_dim_users.csv", header=True, inferSchema=True)

## D.5 Data Exploration
In this section, raw data transformed into DataFrame is checked for exploration

In [13]:
# 1. Data Viewing Top 5
df_order_items.show(5)
df_products.show(5)
df_users.show(5)

+-------------+--------+-------+----------+------------------+----------+
|order_item_id|order_id|user_id|product_id|        sale_price|    status|
+-------------+--------+-------+----------+------------------+----------+
|         3671|    2526|   1976|     14235|0.0199999995529651|  Complete|
|        26255|   18153|  14471|     14235|0.0199999995529651|Processing|
|        69716|   47981|  38538|     14235|0.0199999995529651|Processing|
|       122351|   84256|  67645|     14235|0.0199999995529651|Processing|
|          503|     348|    268|     14235|0.0199999995529651|   Shipped|
+-------------+--------+-------+----------+------------------+----------+
only showing top 5 rows
+----------+--------------------+-----------+-----+----------+------------------+------------------+
|product_id|                name|   category|brand|department|              cost|      retail_price|
+----------+--------------------+-----------+-----+----------+------------------+------------------+
|     1

In [14]:
# 2. Data Viewing Bottom 5
df_order_items.tail(5)
df_products.tail(5)
df_users.tail(5)

[Row(users_id=71364, gender='M', country='France', traffic_source='Search'),
 Row(users_id=29647, gender='M', country='France', traffic_source='Search'),
 Row(users_id=45072, gender='M', country='France', traffic_source='Email'),
 Row(users_id=93769, gender='M', country='France', traffic_source='Search'),
 Row(users_id=6387, gender='M', country='France', traffic_source='Organic')]

In [15]:
# 3. Verify Row Counts
print(f"Total Order Items Raw Rows: {df_order_items.count()}")
print(f"Total Products Raw Rows : {df_products.count()}")
print(f"Total Users Raw Rows : {df_users.count()}")

Total Order Items Raw Rows: 151239
Total Products Raw Rows : 29120
Total Users Raw Rows : 100000


In [16]:
# 4. Ensure all of data type in PySpark align with DDL design before load it into Data Warehouse PostgreSQL
# 4.1 Order Items Table
df_order_items.printSchema()

# 4.2 Products Table
df_products.printSchema()

# 4.3 Users Table
df_users.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- status: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- department: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- retail_price: double (nullable = true)

root
 |-- users_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_source: string (nullable = true)



# E. Data Transformation
In this section, DataFrame is cleaned and transformed using PySpark
1. Filter or clean the missing values
2. Filter where the `status` field = "Complete" in `order_items`
3. Join `order_items` with `products` in PySpark to bring in the `cost` field.
4. Calculate profit AS `profit` using `sale_price` and `cost` ('sale_price - 'cost')
5. The output are: 
    - Fact Table: `fact_sales`
    - Dim Table: `dim_products` and `dim_users`
6. For the record, we need to transform and cleaned dimension table first because:
    - Dimension Table acted as *containers* or parent, otherwise Fact Table acted as *the thing inside container* or child
    - Fact Table contains both foreign key from `products` and `users`, so in order we could find both foreign key, dimension table (`products` and `users` should transformed and load first)

## E.1 Transform `dim_product`

In [17]:
# 1. Filter out rows where `product_id` is NULL (empty)
df_products_no_nulls = df_products.filter(F.col("product_id").isNotNull())
df_products_no_nulls.show(5)

+----------+--------------------+-----------+-----+----------+------------------+------------------+
|product_id|                name|   category|brand|department|              cost|      retail_price|
+----------+--------------------+-----------+-----+----------+------------------+------------------+
|     13842|Low Profile Dyed ...|Accessories|   MG|     Women| 2.518749990849756|              6.25|
|     13928|Low Profile Dyed ...|Accessories|   MG|     Women|2.3383499148894105| 5.949999809265137|
|     14115|Enzyme Regular So...|Accessories|   MG|     Women| 4.879559879379869|10.989999771118164|
|     14157|Enzyme Regular So...|Accessories|   MG|     Women| 4.648769887297898|10.989999771118164|
|     14273|Washed Canvas Ivy...|Accessories|   MG|     Women| 6.507929886473045|15.989999771118164|
+----------+--------------------+-----------+-----+----------+------------------+------------------+
only showing top 5 rows


In [18]:
# 2. Remove duplicated products based on `product_id` field so each product is unique
df_products_unique = df_products_no_nulls.dropDuplicates(["product_id"])
df_products_unique.show(5)

+----------+--------------------+-----------+------------------+----------+------------------+------------+
|product_id|                name|   category|             brand|department|              cost|retail_price|
+----------+--------------------+-----------+------------------+----------+------------------+------------+
|         1|Seven7 Women's Lo...|Tops & Tees|            Seven7|     Women| 27.04799991566688|        49.0|
|         3|Calvin Klein Jean...|Tops & Tees|Calvin Klein Jeans|     Women|41.004999860189855|        69.5|
|         5|Anne Klein Women'...|Tops & Tees|        Anne Klein|     Women|52.357999871484935|        94.0|
|         6|Wilt Women's Colo...|Tops & Tees|              Wilt|     Women| 78.93599966168404|       132.0|
|        12|Jones New York Wo...|Tops & Tees|    Jones New York|     Women| 50.78700002655386|        99.0|
+----------+--------------------+-----------+------------------+----------+------------------+------------+
only showing top 5 rows


In [19]:
# 3. Select required columns, fix data types, and round prices
dim_products = df_products_unique.select(
    "product_id",
    "name",
    "category",
    "brand",
    "department",
    F.round("cost", 2).alias("cost"),
    F.round("retail_price", 2).alias("retail_price")
    )
dim_products.show(10)
dim_products.printSchema()

+----------+--------------------+-----------+------------------+----------+-----+------------+
|product_id|                name|   category|             brand|department| cost|retail_price|
+----------+--------------------+-----------+------------------+----------+-----+------------+
|         1|Seven7 Women's Lo...|Tops & Tees|            Seven7|     Women|27.05|        49.0|
|         3|Calvin Klein Jean...|Tops & Tees|Calvin Klein Jeans|     Women| 41.0|        69.5|
|         5|Anne Klein Women'...|Tops & Tees|        Anne Klein|     Women|52.36|        94.0|
|         6|Wilt Women's Colo...|Tops & Tees|              Wilt|     Women|78.94|       132.0|
|        12|Jones New York Wo...|Tops & Tees|    Jones New York|     Women|50.79|        99.0|
|        13|Jones New York Wo...|Tops & Tees|    Jones New York|     Women|45.94|        76.7|
|        16|2B Paradise Breez...|Tops & Tees|        2b by bebe|     Women|18.42|       32.95|
|        19|Billabong Aussies...|Tops & Tees|     

In [20]:
# 3.1 Calculate total rows
print(f"Total Products Raw Rows : {dim_products.count()}")

Total Products Raw Rows : 29120


In [21]:
# 3.2 Count NULL data
dim_products.select([count(when(col(c).isNull(), c)).alias(c) 
                     for c in dim_products.columns]).show()

+----------+----+--------+-----+----------+----+------------+
|product_id|name|category|brand|department|cost|retail_price|
+----------+----+--------+-----+----------+----+------------+
|         0|   2|       0|   24|         0|   0|           0|
+----------+----+--------+-----+----------+----+------------+



NULL values exist in the `name` field (2 rows) and the `brand` field (24 rows). To preserve data integrity, these rows are not removed; instead, the missing values are imputed with `"Unknown"`.

In [22]:
# 3.3 Filling in NULL data
dim_products_clean = dim_products.na.fill({
    "name": "Unknown",
    "brand": "Unknown"
    })

# 3.4 Count NULL data
dim_products_clean.select([count(when(col(c).isNull(), c)).alias(c) 
                     for c in dim_products_clean.columns]).show()

+----------+----+--------+-----+----------+----+------------+
|product_id|name|category|brand|department|cost|retail_price|
+----------+----+--------+-----+----------+----+------------+
|         0|   0|       0|    0|         0|   0|           0|
+----------+----+--------+-----+----------+----+------------+



In [23]:
# 4. Print schema and inspect sample records
print("--- DIMENSION PRODUCTS TABLE ---") 
dim_products_clean.printSchema()
dim_products_clean.show(5, truncate=False)

--- DIMENSION PRODUCTS TABLE ---
root
 |-- product_id: integer (nullable = true)
 |-- name: string (nullable = false)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = false)
 |-- department: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- retail_price: double (nullable = true)

+----------+------------------------------------------------+-----------+------------------+----------+-----+------------+
|product_id|name                                            |category   |brand             |department|cost |retail_price|
+----------+------------------------------------------------+-----------+------------------+----------+-----+------------+
|1         |Seven7 Women's Long Sleeve Stripe Belted Top    |Tops & Tees|Seven7            |Women     |27.05|49.0        |
|3         |Calvin Klein Jeans Women's Solid Flyaway Shirt  |Tops & Tees|Calvin Klein Jeans|Women     |41.0 |69.5        |
|5         |Anne Klein Women's Plus-Size Button Front Blouse|Tops 

Transformation for `dim_products` is complete. <br>
1. No missing values remain, confirmed by the '0' counts after re-running the NULL check.
2. Metadata updated on printSchema()

## E.2 Transform `dim_users`

In [24]:
# 1. Filter out rows where `users_id is NULL (empty)
df_users_no_nulls = df_users.filter(F.col("users_id").isNotNull())
df_users_no_nulls.show(5)

+--------+------+-------+--------------+
|users_id|gender|country|traffic_source|
+--------+------+-------+--------------+
|   81146|     F| Brasil|        Search|
|   56556|     F| Brasil|        Search|
|    2792|     F| Brasil|        Search|
|   35509|     F| Brasil|        Search|
|   60250|     F| Brasil|        Search|
+--------+------+-------+--------------+
only showing top 5 rows


In [25]:
# 2. Remove duplicated products based on `users_id field so each product is unique
df_users_unique = df_users_no_nulls.dropDuplicates(["users_id"])
df_users_unique.show(5)

+--------+------+-------------+--------------+
|users_id|gender|      country|traffic_source|
+--------+------+-------------+--------------+
|       1|     F|United States|        Search|
|       3|     F|       Brasil|         Email|
|       5|     F|        China|       Organic|
|       6|     M|       Brasil|       Display|
|       9|     M|        China|        Search|
+--------+------+-------------+--------------+
only showing top 5 rows


In [26]:
# 3. Select required columns, fix data types, and round prices
dim_users = df_users_unique.select(
    F.col("users_id").alias("user_id"),  # Change column name
    "gender",
    "country",
    "traffic_source")
dim_users.show(10)
dim_users.printSchema()

+-------+------+-------------+--------------+
|user_id|gender|      country|traffic_source|
+-------+------+-------------+--------------+
|      1|     F|United States|        Search|
|      3|     F|       Brasil|         Email|
|      5|     F|        China|       Organic|
|      6|     M|       Brasil|       Display|
|      9|     M|        China|        Search|
|     12|     M|United States|       Organic|
|     13|     M|  South Korea|        Search|
|     15|     M|United States|       Organic|
|     16|     M|        China|        Search|
|     17|     M|       Brasil|       Organic|
+-------+------+-------------+--------------+
only showing top 10 rows
root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_source: string (nullable = true)



In [27]:
# 3.1 Calculate total rows
print(f"Total Users Raw Rows : {dim_users.count()}")

Total Users Raw Rows : 100000


In [28]:
# 3.2 Count NULL data
dim_users.select([count(when(col(c).isNull(), c)).alias(c) 
                for c in dim_users.columns]).show()

+-------+------+-------+--------------+
|user_id|gender|country|traffic_source|
+-------+------+-------+--------------+
|      0|     0|      0|             0|
+-------+------+-------+--------------+



There are no NULL values in any column.

In [29]:
# 4. Print schema and inspect sample records
print("--- DIMENSION USERS TABLE ---") 
dim_users.printSchema()
dim_users.show(5, truncate=False)

--- DIMENSION USERS TABLE ---
root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_source: string (nullable = true)

+-------+------+-------------+--------------+
|user_id|gender|country      |traffic_source|
+-------+------+-------------+--------------+
|1      |F     |United States|Search        |
|3      |F     |Brasil       |Email         |
|5      |F     |China        |Organic       |
|6      |M     |Brasil       |Display       |
|9      |M     |China        |Search        |
+-------+------+-------------+--------------+
only showing top 5 rows


Transformation for `dim_users` is complete. <br>
1. No missing values, confirmed by the '0' counts in the NULL check.

## E.3 Transform `fact_sales`

In [30]:
# 1. Keep only completed order transactions (status == "Complete")
df_order_items_completed = df_order_items.filter(F.col("status") == "Complete")
df_order_items_completed.show(5)
print(f"Total Order Raw Rows : {df_order_items_completed.count()}")

+-------------+--------+-------+----------+------------------+--------+
|order_item_id|order_id|user_id|product_id|        sale_price|  status|
+-------------+--------+-------+----------+------------------+--------+
|         3671|    2526|   1976|     14235|0.0199999995529651|Complete|
|       125525|   86484|  69452|     14159|0.4900000095367431|Complete|
|        44698|   30846|  24737|     28700|               1.5|Complete|
|        84020|   57915|  46508|     28700|               1.5|Complete|
|        84874|   58502|  46991|     28700|               1.5|Complete|
+-------------+--------+-------+----------+------------------+--------+
only showing top 5 rows
Total Order Raw Rows : 38097


In [31]:
# 2. Filter out NULL order_item_id and drop duplicate records
df_order_items_clean = df_order_items_completed\
    .filter(F.col("order_item_id").isNotNull()).dropDuplicates(["order_item_id"])

df_order_items_clean.show(5)
print(f"Total Order Item ID Raw Rows : {df_order_items_clean.count()}")

+-------------+--------+-------+----------+------------------+--------+
|order_item_id|order_id|user_id|product_id|        sale_price|  status|
+-------------+--------+-------+----------+------------------+--------+
|            7|       5|      3|      9694|  39.9900016784668|Complete|
|           10|       7|      5|       440|  33.9900016784668|Complete|
|           15|      10|      7|     12680|10.989999771118164|Complete|
|           22|      16|     12|     24465|              28.5|Complete|
|           30|      21|     16|     24949|              33.0|Complete|
+-------------+--------+-------+----------+------------------+--------+
only showing top 5 rows
Total Order Item ID Raw Rows : 38097


In [32]:
# 3. Select 'cost' columns in order to calculate 'profit'
df_product_costs = df_products.select("product_id", "cost")

In [33]:
# 4. Join order items with product_costs to retrieve 'cost'
df_sales_joined = df_order_items_clean.join(df_product_costs, on="product_id", how="inner")

In [34]:
# 5. Calculate gross profit (sale_price - cost)
df_sales_profit = df_sales_joined.withColumn("profit", F.col("sale_price") - F.col("cost"))

df_sales_profit.show(5)
print(f"Total Profit Raw Rows : {df_sales_profit.count()}")

+----------+-------------+--------+-------+------------------+--------+------------------+------------------+
|product_id|order_item_id|order_id|user_id|        sale_price|  status|              cost|            profit|
+----------+-------------+--------+-------+------------------+--------+------------------+------------------+
|      9694|            7|       5|      3|  39.9900016784668|Complete| 22.47438105175168|17.515620626715116|
|       440|           10|       7|      5|  33.9900016784668|Complete| 19.10238090094307|14.887620777523725|
|     12680|           15|      10|      7|10.989999771118164|Complete|5.4840098839865625|5.5059898871316015|
|     24465|           22|      16|     12|              28.5|Complete|15.988500019535422|12.511499980464578|
|     24949|           30|      21|     16|              33.0|Complete|21.417000006884336|11.582999993115664|
+----------+-------------+--------+-------+------------------+--------+------------------+------------------+
only showi

In [35]:
# 6. Select final columns/data and round numeric values up to 2 decimal places
fact_sales = df_sales_profit.select(
    "order_item_id",
    "product_id",
    "user_id",
    F.round("sale_price", 2).alias("sale_price"),
    F.round("cost", 2).alias("cost"),
    F.round("profit", 2).alias("profit")
    )
fact_sales.show(5)
fact_sales.printSchema()

+-------------+----------+-------+----------+----+------+
|order_item_id|product_id|user_id|sale_price|cost|profit|
+-------------+----------+-------+----------+----+------+
|         6357|     28774|   3440|      2.78|0.99|  1.79|
|        38723|     15830|  21448|      4.99|2.73|  2.26|
|        72702|     14246|  40227|      5.69|2.33|  3.36|
|        66176|      7156|  36591|      6.49|2.82|  3.67|
|       141952|      9464|  78560|       6.5|2.75|  3.75|
+-------------+----------+-------+----------+----+------+
only showing top 5 rows
root
 |-- order_item_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- profit: double (nullable = true)



In [36]:
# 6.1 Calculate total rows
print(f"Total Sales Raw Rows : {fact_sales.count()}")

Total Sales Raw Rows : 38097


In [37]:
# 6.2 Count NULL data
fact_sales.select([count(when(col(c).isNull(), c)).alias(c) 
                for c in fact_sales.columns]).show()

+-------------+----------+-------+----------+----+------+
|order_item_id|product_id|user_id|sale_price|cost|profit|
+-------------+----------+-------+----------+----+------+
|            0|         0|      0|         0|   0|     0|
+-------------+----------+-------+----------+----+------+



There are no NULL values in any column.

In [38]:
# 7. Print schema and inspect sample records
print("--- FACT SALES TABLE ---") 
fact_sales.printSchema()
fact_sales.show(5, truncate=False)

--- FACT SALES TABLE ---
root
 |-- order_item_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- profit: double (nullable = true)

+-------------+----------+-------+----------+----+------+
|order_item_id|product_id|user_id|sale_price|cost|profit|
+-------------+----------+-------+----------+----+------+
|6357         |28774     |3440   |2.78      |0.99|1.79  |
|38723        |15830     |21448  |4.99      |2.73|2.26  |
|72702        |14246     |40227  |5.69      |2.33|3.36  |
|66176        |7156      |36591  |6.49      |2.82|3.67  |
|141952       |9464      |78560  |6.5       |2.75|3.75  |
+-------------+----------+-------+----------+----+------+
only showing top 5 rows


Transformation for `fact_sales` is complete. <br>
1. No missing values, confirmed by the '0' counts in the NULL check.

# F. Data Loading
In this final stage, we will loading our cleaned PySpark DataFrames into PostgreSQL Data Warehouse with following this steps:
1. Load Dimension Tables (`dim_products_clean` and `dim_users`)
2. Load Fact Table (`fact_sales`)

In [ ]:
# 1. Define PostgreSQL Database Connection Parameters
# Credentials are loaded from a local .env file (not committed to GitHub).
# See .env.example for the required variable names.
load_dotenv()
DB_USER     = os.getenv("PG_USER")
DB_PASSWORD = os.getenv("PG_PASSWORD")
DB_HOST     = os.getenv("PG_HOST")
DB_PORT     = os.getenv("PG_PORT")
DB_NAME     = os.getenv("PG_DB")

In [ ]:
# 2. Install packages for the Pandas → PostgreSQL load
!pip -q install sqlalchemy psycopg2

In [42]:
from sqlalchemy import create_engine

# 3. Create connection SQLAlchemy Engine using .env credentials
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# 4. Convert PySpark DataFrames to Pandas DataFrames
print("Converting PySpark DataFrames to Pandas...")
df_products_pd = dim_products_clean.toPandas()
df_users_pd = dim_users.toPandas()
df_sales_pd = fact_sales.toPandas()

# 5. Load into PostgreSQL (dimensions first, then fact table to satisfy foreign keys)
print("Loading dim_products...")
df_products_pd.to_sql("dim_products", engine, if_exists="append", index=False)
print("dim_products loaded successfully!")

print("Loading dim_users...")
df_users_pd.to_sql("dim_users", engine, if_exists="append", index=False)
print("dim_users loaded successfully!")

print("Loading fact_sales...")
df_sales_pd.to_sql("fact_sales", engine, if_exists="append", index=False)
print("All data loaded into PostgreSQL successfully!")

Converting PySpark DataFrames to Pandas...
Loading dim_products...
dim_products loaded successfully!
Loading dim_users...
dim_users loaded successfully!
Loading fact_sales...
All data loaded into PostgreSQL successfully!


In [43]:
# 5. Stop Spark Session
spark.stop()

# G. Insight & Key Findings
1. Comparison of total rows before and after transformation:  

| Target Table | Source Table | Before | After | Difference | Description |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `fact_sales` | `order_items` | 151,239 | 38,097 | −113,142 | Kept only `status = "Complete"` orders, so incomplete transactions are excluded from sales analysis. |
| `dim_products` | `products` | 29,120 | 29,120 | 0 | No rows dropped; primary key had no nulls and no duplicates. |
| `dim_users` | `users` | 100,000 | 100,000 | 0 | No rows dropped; primary key had no nulls and no duplicates. |

2. NULL values were found in the `dim_products` table:
    - `name`: 2 rows null
    - `brand`: 24 rows null
    - To preserve data integrity and avoid dropping otherwise valid products, these nulls were imputed with `"Unknown"` instead of removing the rows.
3. No NULL values were found in the `dim_users` or `fact_sales` tables.

The presentation link below:
https://docs.google.com/presentation/d/1PadFn-eaaY8wAZoK1A5azANObLGVlsKU/edit?usp=sharing&ouid=111269027971600279535&rtpof=true&sd=true